In [ ]:
#Alexis #George
#https://www.kaggle.com/datasets/sujay1844/used-car-prices

In [ ]:
import xgboost as xgb # XGB model for regression
from sklearn.model_selection import train_test_split  #Splitting the data to testing and training sets.
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score #Metrics to check how good the model is
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
df=pd.read_csv("usedcar_prices.csv") #Load dataset
df.head() #Preview first few rows


,Unnamed: 0,Name,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,New_Price,Price
0,1,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,Diesel,Manual,First,19.67 kmpl,1582 CC,126.2 bhp,5.0,NaN,12.50
1,2,Honda Jazz V,Chennai,2011,46000,Petrol,Manual,First,13 km/kg,1199 CC,88.7 bhp,5.0,8.61 Lakh,4.50
2,3,Maruti Ertiga VDI,Chennai,2012,87000,Diesel,Manual,First,20.77 kmpl,1248 CC,88.76 bhp,7.0,NaN,6.00
3,4,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670,Diesel,Automatic,Second,15.2 kmpl,1968 CC,140.8 bhp,5.0,NaN,17.74
4,6,Nissan Micra Diesel XV,Jaipur,2013,86999,Diesel,Manual,First,23.08 kmpl,1461 CC,63.1 bhp,5.0,NaN,3.50


In [ ]:
df = df.drop(["Unnamed: 0"], axis=1) #removed unnamed column because its just an index and not useful or prediction
df = df.drop(["New_Price"], axis=1) #removed new_price since there are too many NAN values and not needed

df

,Name,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,Price
0,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,Diesel,Manual,First,19.67 kmpl,1582 CC,126.2 bhp,5.0,12.50
1,Honda Jazz V,Chennai,2011,46000,Petrol,Manual,First,13 km/kg,1199 CC,88.7 bhp,5.0,4.50
2,Maruti Ertiga VDI,Chennai,2012,87000,Diesel,Manual,First,20.77 kmpl,1248 CC,88.76 bhp,7.0,6.00
3,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670,Diesel,Automatic,Second,15.2 kmpl,1968 CC,140.8 bhp,5.0,17.74
4,Nissan Micra Diesel XV,Jaipur,2013,86999,Diesel,Manual,First,23.08 kmpl,1461 CC,63.1 bhp,5.0,3.50
...,...,...,...,...,...,...,...,...,...,...,...,...
5842,Maruti Swift VDI,Delhi,2014,27365,Diesel,Manual,First,28.4 kmpl,1248 CC,74 bhp,5.0,4.75
5843,Hyundai Xcent 1.1 CRDi S,Jaipur,2015,100000,Diesel,Manual,First,24.4 kmpl,1120 CC,71 bhp,5.0,4.00
5844,Mahindra Xylo D4 BSIV,Jaipur,2012,55000,Diesel,Manual,Second,14.0 kmpl,2498 CC,112 bhp,8.0,2.90
5845,Maruti Wagon R VXI,Kolkata,2013,46000,Petrol,Manual,First,18.9 kmpl,998 CC,67.1 bhp,5.0,2.65


In [ ]:
#Check for missing values
df.isnull().sum()

,0
Unnamed: 0,0
Name,0
Location,0
Year,0
Kilometers_Driven,0
Fuel_Type,0
Transmission,0
Owner_Type,0
Mileage,2
Engine,36


In [ ]:
df = df.dropna() #Remove rows with missing values to clean the dataset
df

,Name,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,Price
0,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,Diesel,Manual,First,19.67 kmpl,1582 CC,126.2 bhp,5.0,12.50
1,Honda Jazz V,Chennai,2011,46000,Petrol,Manual,First,13 km/kg,1199 CC,88.7 bhp,5.0,4.50
2,Maruti Ertiga VDI,Chennai,2012,87000,Diesel,Manual,First,20.77 kmpl,1248 CC,88.76 bhp,7.0,6.00
3,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670,Diesel,Automatic,Second,15.2 kmpl,1968 CC,140.8 bhp,5.0,17.74
4,Nissan Micra Diesel XV,Jaipur,2013,86999,Diesel,Manual,First,23.08 kmpl,1461 CC,63.1 bhp,5.0,3.50
...,...,...,...,...,...,...,...,...,...,...,...,...
5842,Maruti Swift VDI,Delhi,2014,27365,Diesel,Manual,First,28.4 kmpl,1248 CC,74 bhp,5.0,4.75
5843,Hyundai Xcent 1.1 CRDi S,Jaipur,2015,100000,Diesel,Manual,First,24.4 kmpl,1120 CC,71 bhp,5.0,4.00
5844,Mahindra Xylo D4 BSIV,Jaipur,2012,55000,Diesel,Manual,Second,14.0 kmpl,2498 CC,112 bhp,8.0,2.90
5845,Maruti Wagon R VXI,Kolkata,2013,46000,Petrol,Manual,First,18.9 kmpl,998 CC,67.1 bhp,5.0,2.65


In [ ]:
#Confirming dataset no longer has missing values
df.isnull().sum()

,0
Name,0
Location,0
Year,0
Kilometers_Driven,0
Fuel_Type,0
Transmission,0
Owner_Type,0
Mileage,0
Engine,0
Power,0


In [ ]:
#Remove non numeric columns since model requires numerical inputs
df = df.drop(["Location", "Name", "Fuel_Type", "Transmission", "Owner_Type", "Mileage", "Engine", "Power"], axis=1)
df.head() #Verify cleaned dataset

,Year,Kilometers_Driven,Seats,Price
0,2015,41000,5.0,12.50
1,2011,46000,5.0,4.50
2,2012,87000,7.0,6.00
3,2013,40670,5.0,17.74
4,2013,86999,5.0,3.50


In [ ]:
X = df.drop("Price", axis=1)
y = df["Price"] #Target variable, price we are predicting

X_simplified = X
y_simplified = y

In [ ]:
#80% Training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X_simplified, y_simplified,
    test_size=0.2,
    random_state=42
)

In [ ]:
model = xgb.XGBRegressor(
    max_depth=4, #Controls how deep each tree can go
    n_estimators=50, #number of trees used in the model
    learning_rate=0.1, #how quickly the model updates during training
    objective='reg:squarederror', #loss function used for regression
    random_state=42, #Ensures same results every time
    subsample=0.8, #80% data for each tree
    colsample_bytree=0.8 #80% features for each tree
)

model.fit(X_train, y_train, verbose=False) #Trains the model using the training data

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=50,
             n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
y_train_pred = model.predict(X_train) #Predictions on training data
y_test_pred = model.predict(X_test) #Predictions on testing data

In [ ]:
mse = mean_squared_error(y_test, y_test_pred) #Calculates squared error between actual and predicted values
mae = mean_absolute_error(y_test, y_test_pred) #Calculates average difference between actual and predicted values
r2 = r2_score(y_test, y_test_pred) #Shows how well the model fits the data

print("Mean Squared Error:", mse)
print("Mean Absolute Error:", mae)
print("R2 Score:", r2)

Mean Squared Error: 91.30739179914335
Mean Absolute Error: 6.309875581145492
R2 Score: 0.19216600316448373


For this lab I used a used car dataset to build a regression model that predicts car prices based on features like the year, kilometers driven, and number of seats. I started by cleaning the dataset, removing unnecessary columns such as names, locations, and other text fields and also removing all missing values by dropping incomplete rows. I Then split the data into training and testing sets and applied XGBoost regressor model. I used parameters such as max depth, number of estimators, and leaerning rate, and also tested different values to see how they affected performance. After training the model, I got some predictions and evaluated the results using Mean Squared Error, Mean Absolute Error, and R2 score. Results show MSE: 91.30, MAE: 6.30, and R2 Score: 0.1921